In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/fraudTrain.csv")
df = df.drop(columns=['Unnamed: 0'])
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['dob'] = pd.to_datetime(df['dob'])
df.shape

(1296675, 22)

In [3]:
df['hour']= df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['is_night'] = df['hour'].isin([22,23,0,1,2,3]).astype(int)


In [4]:
df['age']= (df['trans_date_trans_time'] - df['dob']).dt.days // 365

In [5]:
df = df.sort_values(['cc_num','trans_date_trans_time'])
df['cc_txn_total_count'] = df.groupby('cc_num').cumcount() + 1
df['cc_avg_amt'] = df.groupby('cc_num')['amt'].transform(lambda x : x.expanding().mean().shift(1))
df['amt_vs_cc_avg'] = df['amt'] / df['cc_avg_amt']


In [6]:
df[['cc_num', 'trans_date_trans_time', 'amt', 'cc_txn_total_count', 'cc_avg_amt', 'amt_vs_cc_avg']].head(10)

,cc_num,trans_date_trans_time,amt,cc_txn_total_count,cc_avg_amt,amt_vs_cc_avg
1017,60416207185,2019-01-01 12:47:15,7.27,1,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,2,7.270000,7.281981
2726,60416207185,2019-01-02 08:47:36,82.08,3,30.105000,2.726457
2882,60416207185,2019-01-02 12:38:14,34.79,4,47.430000,0.733502
2907,60416207185,2019-01-02 13:10:46,27.18,5,44.270000,0.613960
4135,60416207185,2019-01-03 13:56:35,6.87,6,40.852000,0.168168
4337,60416207185,2019-01-03 17:05:10,8.43,7,35.188333,0.239568
5467,60416207185,2019-01-04 13:59:55,117.11,8,31.365714,3.733695
6027,60416207185,2019-01-04 21:17:22,26.74,9,42.083750,0.635400
6273,60416207185,2019-01-05 00:42:24,105.20,10,40.378889,2.605322


In [7]:
df['mins_since_last_txn'] = df.groupby('cc_num')['trans_date_trans_time'].diff().dt.total_seconds() / 60

In [8]:
df[['cc_num', 'trans_date_trans_time', 'amt', 'mins_since_last_txn']].head(10)

,cc_num,trans_date_trans_time,amt,mins_since_last_txn
1017,60416207185,2019-01-01 12:47:15,7.27,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,1197.700000
2726,60416207185,2019-01-02 08:47:36,82.08,2.650000
2882,60416207185,2019-01-02 12:38:14,34.79,230.633333
2907,60416207185,2019-01-02 13:10:46,27.18,32.533333
4135,60416207185,2019-01-03 13:56:35,6.87,1485.816667
4337,60416207185,2019-01-03 17:05:10,8.43,188.583333
5467,60416207185,2019-01-04 13:59:55,117.11,1254.750000
6027,60416207185,2019-01-04 21:17:22,26.74,437.450000
6273,60416207185,2019-01-05 00:42:24,105.20,205.033333


In [10]:
df[['cc_num', 'trans_date_trans_time', 'amt', 'cc_avg_amt', 'cc_avg_amt_roll10']].head(15)

,cc_num,trans_date_trans_time,amt,cc_avg_amt,cc_avg_amt_roll10
1017,60416207185,2019-01-01 12:47:15,7.27,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,7.270000,7.270000
2726,60416207185,2019-01-02 08:47:36,82.08,30.105000,30.105000
2882,60416207185,2019-01-02 12:38:14,34.79,47.430000,47.430000
2907,60416207185,2019-01-02 13:10:46,27.18,44.270000,44.270000
4135,60416207185,2019-01-03 13:56:35,6.87,40.852000,40.852000
4337,60416207185,2019-01-03 17:05:10,8.43,35.188333,35.188333
5467,60416207185,2019-01-04 13:59:55,117.11,31.365714,31.365714
6027,60416207185,2019-01-04 21:17:22,26.74,42.083750,42.083750
6273,60416207185,2019-01-05 00:42:24,105.20,40.378889,40.378889


In [12]:
df['cc_median_amt_roll10'] = df.groupby('cc_num')['amt'].transform(
    lambda x: x.rolling(window=10, min_periods=1).median().shift(1)
)
df['amt_vs_cc_median_roll10'] = df['amt'] / df['cc_median_amt_roll10']

In [14]:
df[['cc_num', 'trans_date_trans_time', 'amt', 'cc_median_amt_roll10', 'amt_vs_cc_median_roll10']].head(15)

,cc_num,trans_date_trans_time,amt,cc_median_amt_roll10,amt_vs_cc_median_roll10
1017,60416207185,2019-01-01 12:47:15,7.27,NaN,NaN
2724,60416207185,2019-01-02 08:44:57,52.94,7.270,7.281981
2726,60416207185,2019-01-02 08:47:36,82.08,30.105,2.726457
2882,60416207185,2019-01-02 12:38:14,34.79,52.940,0.657159
2907,60416207185,2019-01-02 13:10:46,27.18,43.865,0.619628
4135,60416207185,2019-01-03 13:56:35,6.87,34.790,0.197471
4337,60416207185,2019-01-03 17:05:10,8.43,30.985,0.272067
5467,60416207185,2019-01-04 13:59:55,117.11,27.180,4.308683
6027,60416207185,2019-01-04 21:17:22,26.74,30.985,0.862998
6273,60416207185,2019-01-05 00:42:24,105.20,27.180,3.870493


In [15]:
nan_cols = ['cc_avg_amt', 'cc_avg_amt_roll10', 'cc_median_amt_roll10', 
            'amt_vs_cc_avg', 'amt_vs_cc_avg_roll10', 'amt_vs_cc_median_roll10',
            'mins_since_last_txn']
df[nan_cols].isnull().sum()

cc_avg_amt                 983
cc_avg_amt_roll10          983
cc_median_amt_roll10       983
amt_vs_cc_avg              983
amt_vs_cc_avg_roll10       983
amt_vs_cc_median_roll10    983
mins_since_last_txn        983
dtype: int64

In [16]:
df['cc_num'].nunique()

983

In [17]:
overall_avg_amt = df['amt'].mean()

df['cc_avg_amt'] = df['cc_avg_amt'].fillna(overall_avg_amt)
df['cc_avg_amt_roll10'] = df['cc_avg_amt_roll10'].fillna(overall_avg_amt)
df['cc_median_amt_roll10'] = df['cc_median_amt_roll10'].fillna(overall_avg_amt)

df['amt_vs_cc_avg'] = df['amt_vs_cc_avg'].fillna(1)
df['amt_vs_cc_avg_roll10'] = df['amt_vs_cc_avg_roll10'].fillna(1)
df['amt_vs_cc_median_roll10'] = df['amt_vs_cc_median_roll10'].fillna(1)

df['mins_since_last_txn'] = df['mins_since_last_txn'].fillna(df['mins_since_last_txn'].max())

In [18]:
df[nan_cols].isnull().sum()

cc_avg_amt                 0
cc_avg_amt_roll10          0
cc_median_amt_roll10       0
amt_vs_cc_avg              0
amt_vs_cc_avg_roll10       0
amt_vs_cc_median_roll10    0
mins_since_last_txn        0
dtype: int64

In [19]:
df['gender_encoded']= df['gender'].map({'M':0 , 'F':1})

In [20]:
df = pd.get_dummies(df, columns=['category'], prefix='category', drop_first=True)

In [21]:
merchant_counts = df['merchant'].value_counts()
df['merchant_freq'] = df['merchant'].map(merchant_counts)

In [22]:
df.columns.tolist()

['trans_date_trans_time',
 'cc_num',
 'merchant',
 'amt',
 'first',
 'last',
 'gender',
 'street',
 'city',
 'state',
 'zip',
 'lat',
 'long',
 'city_pop',
 'job',
 'dob',
 'trans_num',
 'unix_time',
 'merch_lat',
 'merch_long',
 'is_fraud',
 'hour',
 'day_of_week',
 'is_night',
 'age',
 'cc_txn_total_count',
 'cc_avg_amt',
 'amt_vs_cc_avg',
 'mins_since_last_txn',
 'cc_avg_amt_roll10',
 'amt_vs_cc_avg_roll10',
 'cc_median_amt_roll10',
 'amt_vs_cc_median_roll10',
 'gender_encoded',
 'category_food_dining',
 'category_gas_transport',
 'category_grocery_net',
 'category_grocery_pos',
 'category_health_fitness',
 'category_home',
 'category_kids_pets',
 'category_misc_net',
 'category_misc_pos',
 'category_personal_care',
 'category_shopping_net',
 'category_shopping_pos',
 'category_travel',
 'merchant_freq']

In [23]:
'gender_encoded' in df.columns

True

In [24]:
columns_to_drop = [
    'street',        
    'first',         
    'last',         
    'zip',           
    'trans_num',     
    'city',          
    'state',         
    'job',          
    'merchant',      
    'gender',       
    'dob',           #
    'trans_date_trans_time',  
    'cc_num',        
]

df_model = df.drop(columns=columns_to_drop)
df_model.shape

(1296675, 35)

In [25]:
df_model.info()
df_model.isnull().sum().sum()

<class 'pandas.DataFrame'>
Index: 1296675 entries, 1017 to 1296427
Data columns (total 35 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   amt                      1296675 non-null  float64
 1   lat                      1296675 non-null  float64
 2   long                     1296675 non-null  float64
 3   city_pop                 1296675 non-null  int64  
 4   unix_time                1296675 non-null  int64  
 5   merch_lat                1296675 non-null  float64
 6   merch_long               1296675 non-null  float64
 7   is_fraud                 1296675 non-null  int64  
 8   hour                     1296675 non-null  int32  
 9   day_of_week              1296675 non-null  int32  
 10  is_night                 1296675 non-null  int64  
 11  age                      1296675 non-null  int64  
 12  cc_txn_total_count       1296675 non-null  int64  
 13  cc_avg_amt               1296675 non-null  float64
 14 

np.int64(0)

In [26]:
df_model.to_csv('../data/fraud_model_ready.csv', index=False)
df.to_csv('../data/fraud_full_with_features.csv', index=False)